In [1]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from src.features import load_and_clean, add_features, get_splits, make_preprocessor

df = add_features(load_and_clean("../data/raw/telco_churn.csv"))
for col in ["addon_count", "has_support_addons", "is_new_customer"]:
    print(df.groupby(col)["churn"].agg(["mean", "count"]).round(3), "\n")

              mean  count
addon_count              
0            0.214   2219
1            0.458    966
2            0.358   1033
3            0.274   1118
4            0.223    852
5            0.124    571
6            0.053    284 

                     mean  count
has_support_addons              
0                   0.334   4079
1                   0.171   2964 

                  mean  count
is_new_customer              
0                0.171   4857
1                0.474   2186 



In [2]:
X_train, X_test, y_train, y_test = get_splits("../data/raw/telco_churn.csv")
print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(5634, 22) (1409, 22)
0.2654 0.2654


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

def cv_report(name, pipe):
    s = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, return_train_score=True)
    return {"model": name, "train_AUC": s["train_roc_auc"].mean(),
            "CV_AUC": s["test_roc_auc"].mean(), "CV_AUC_std": s["test_roc_auc"].std(),
            "CV_PR_AUC": s["test_pr_auc"].mean(), "CV_PR_std": s["test_pr_auc"].std()}

rows = [cv_report("Dummy", Pipeline([("pre", make_preprocessor()),
                                     ("clf", DummyClassifier(strategy="prior"))]))]
for C in [0.001, 0.01, 0.1, 1, 10, 100]:   # small C = strong regularization
    pipe = Pipeline([("pre", make_preprocessor()),
                     ("clf", LogisticRegression(C=C, max_iter=2000))])
    rows.append(cv_report(f"LogReg C={C}", pipe))

cv_table = pd.DataFrame(rows).round(4)
cv_table

,model,train_AUC,CV_AUC,CV_AUC_std,CV_PR_AUC,CV_PR_std
0,Dummy,0.5000,0.5000,0.0000,0.2654,0.0001
1,LogReg C=0.001,0.8419,0.8409,0.0106,0.6524,0.0138
2,LogReg C=0.01,0.8480,0.8455,0.0114,0.6629,0.0155
3,LogReg C=0.1,0.8503,0.8464,0.0112,0.6630,0.0151
4,LogReg C=1,0.8507,0.8464,0.0114,0.6619,0.0147
5,LogReg C=10,0.8509,0.8465,0.0115,0.6608,0.0145
6,LogReg C=100,0.8509,0.8464,0.0116,0.6606,0.0145


In [4]:
best_C = 0.1   # chosen from cv_table: tied with larger C, but most regularized
print(cv_table.loc[cv_table["model"] == f"LogReg C={best_C}"])

          model  train_AUC  CV_AUC  CV_AUC_std  CV_PR_AUC  CV_PR_std
3  LogReg C=0.1     0.8503  0.8464      0.0112      0.663     0.0151
